# IOAI — 2026 Contest Ticket Extraction (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-ticket-extraction/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Customer Ticket Extraction — 고객 문의 정보 추출 (KOAI 2026)

한국어 고객 문의 텍스트에서 **4개 필드**를 추출한다: `category`(billing/technical/account/shipping/refund),
`priority`(low/medium/high), `product`(제품명, 자유형), `sentiment`(negative/neutral/positive).
`data/train.csv`(id,text + 4필드 라벨)로 학습, `data/test.csv`(id,text)를 예측해 **평탄화** `submission.csv`
(`row_id, value`, row_id=`<티켓id>__<필드>`)를 만든다. 평가: 전체 (row_id,value) **정확도**.
원 대회는 Qwen2.5-7B(LLM) baseline 을 주지만, 라벨 600개가 있으므로 **지도학습**이 더 빠르고 정확하다.

In [ ]:
import pandas as pd
train = pd.read_csv("data/train.csv")     # id, text, category, priority, product, sentiment
test  = pd.read_csv("data/test.csv")      # id, text
FIELDS = ["category", "priority", "product", "sentiment"]
print("train", len(train), "| test", len(test))

In [ ]:
# 베이스라인: 전부 unknown — 정확도 0
rows = [{"row_id": f"{r.id}__{f}", "value": "unknown"} for _, r in test.iterrows() for f in FIELDS]
pd.DataFrame(rows).to_csv("submission.csv", index=False)
print("saved submission.csv", len(rows))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)